# Issue Writer — Gemma 4 (E4B) + Unsloth + QLoRA

Veri seti: [`fport/issue-writer-tr-en`](https://huggingface.co/datasets/fport/issue-writer-tr-en)
— 13.000 örnek, %50 Türkçe / %50 İngilizce, çıktı her zaman geçerli JSON.

### Önceki denemeden farklar

| | Eski notebook | Bu notebook |
|---|---|---|
| Eğitim uzunluğu | `max_steps=60` → 480 örnek (**%1**) | `num_train_epochs=2` → ~21.900 örnek |
| LoRA kapasitesi | `r=16, alpha=16` | `r=32, alpha=64` |
| Değerlendirme | yok | eval split + test setinde metrik |
| Maske kontrolü | yok | eğitimden önce doğrulanıyor |
| Öğrenme oranı | `2e-4` sabit | `1e-4` + cosine + warmup |
| Veri | genel talimat seti | göreve özel, yapılandırılmış çıktı |

**60 adım bir demoydu, eğitim değil.** Başarısızlığın ana sebebi buydu.

### Süre (Colab Pro)

| GPU | Model | 2 epoch |
|---|---|---|
| L4 24GB | gemma-4-E4B | ~2–2,5 saat |
| A100 40GB | gemma-4-E4B | ~1–1,5 saat |
| A100 40GB | gemma-4-31B (QLoRA) | ~6–8 saat |


## 1 — GPU


In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'GPU yok: Runtime > Change runtime type > GPU'
VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
GPU  = torch.cuda.get_device_name(0)
print(f'{GPU} · {VRAM:.0f} GB · bf16 {torch.cuda.is_bf16_supported()}')


NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07
NVIDIA A100-SXM4-80GB · 79 GB · bf16 True


In [5]:
print(f'{GPU} · {VRAM:.0f} GB · bf16 {torch.cuda.is_bf16_supported()}')

NVIDIA A100-SXM4-80GB · 79 GB · bf16 True


## 2 — Unsloth kurulumu

Kurulumdan sonra Colab "restart session" derse **restart etme**, devam et.
Sürüm çakışması alırsan en alttaki *Sorun giderme* bölümünde pinli alternatif var.


In [2]:
%%capture
import os
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --upgrade --no-cache-dir unsloth unsloth_zoo
    !pip install -q sentencepiece protobuf hf_transfer 'huggingface_hub>=0.34'


In [3]:
import unsloth, transformers, trl, torch
print('unsloth', unsloth.__version__, '| transformers', transformers.__version__,
      '| trl', trl.__version__, '| torch', torch.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.9.2 | transformers 5.5.0 | trl 0.24.0 | torch 2.11.0+cu128


## 3 — Hugging Face girişi

Gemma lisansını [model sayfasından](https://huggingface.co/google/gemma-4-E4B-it) bir kez
kabul etmen gerekiyor.

Token dört kaynaktan sırayla aranır, ilk bulunan kullanılır:

1. `HF_TOKEN` ortam değişkeni
2. Colab *Secrets* (yalnızca tarayıcıdaki Colab'da çalışır)
3. `huggingface-cli login` ile daha önce kaydedilmiş token
4. elle giriş (yazarken ekranda görünmez)

> **VS Code / uzak makine kullanıyorsan:** Colab Secrets çalışmaz — `userdata.get()`
> tarayıcıda bir izin penceresi açar, o pencere olmadığı için çağrı zaman aşımına düşer.
> Bu hücre bunu algılayıp 3. veya 4. yola geçer. Kalıcı çözüm için bir kez şunu çalıştır:
> `huggingface-cli login`


In [6]:
import os


def get_hf_token() -> str:
    """Token'i ortamdan bagimsiz sekilde bulur.

    Colab Secrets yalnizca tarayicidaki Colab'da calisir: userdata.get() bir izin
    penceresi acar ve VS Code / uzak kernel baglantisinda o pencere gosterilemedigi
    icin cagri TimeoutException ile duser. Bu yuzden sirayla deniyoruz.
    """
    tok = os.environ.get('HF_TOKEN')
    if tok:
        print('kaynak: HF_TOKEN ortam degiskeni')
        return tok

    try:                                    # 2) tarayicidaki Colab
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
        if tok:
            print('kaynak: Colab Secrets')
            return tok
    except Exception as e:
        print(f'Colab Secrets kullanilamadi ({type(e).__name__}), devam ediliyor')

    try:                                    # 3) huggingface-cli login
        from huggingface_hub import get_token
        tok = get_token()
        if tok:
            print('kaynak: kayitli huggingface-cli oturumu')
            return tok
    except Exception:
        pass

    from getpass import getpass           # 4) elle
    return getpass('HF token (ekranda gorunmez): ').strip()


os.environ['HF_TOKEN'] = get_hf_token()
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

from huggingface_hub import whoami
print('giris:', whoami()['name'])


Colab Secrets kullanilamadi (TimeoutException), devam ediliyor
giris: fport


## 4 — Model

`E4B` = etkin 4,5B parametre (toplam 8B, Per-Layer Embedding mimarisi), 128K bağlam.
VRAM 35GB üzerindeyse notebook 31B'ye çıkar; istemiyorsan `MODEL` satırını elle yaz.


In [7]:
from unsloth import FastModel

MODEL  = 'unsloth/gemma-4-E4B-it'
MAXLEN = 2048

# 4-bit bir BELLEK COZUMUDUR, hiz cozumu degil. 40GB ustu VRAM'de gereksiz ve
# yavastir: her matris carpiminda dequantization maliyeti odenir. bf16 hem daha
# hizli hem daha dogru.
LOAD_4BIT = VRAM < 40

# Efektif batch her durumda 16; degisen sadece GPU'yu ne kadar doldurdugumuz.
if VRAM >= 70:   BATCH, ACCUM = 8, 2     # A100 80GB
elif VRAM >= 35: BATCH, ACCUM = 4, 4     # A100 40GB
elif VRAM >= 22: BATCH, ACCUM = 2, 8     # L4 24GB
else:            BATCH, ACCUM = 1, 16    # T4 16GB

model, tokenizer = FastModel.from_pretrained(
    model_name     = MODEL,
    max_seq_length = MAXLEN,
    load_in_4bit   = LOAD_4BIT,
    load_in_8bit   = False,
    full_finetuning= False,
    token          = os.environ['HF_TOKEN'],
)
print(f"{model.config.model_type} | {'4-bit' if LOAD_4BIT else 'bf16'} "
      f"| efektif batch {BATCH * ACCUM} | VRAM {VRAM:.0f} GB")


==((====))==  Unsloth 2026.9.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

gemma4 | efektif batch 16


## 5 — LoRA

`r=32, alpha=64`. Eski denemedeki `r=16, alpha=16` bu iş için dar: model sadece
üslup değil, **sabit bir JSON şeması** öğrenmek zorunda. `alpha = 2r` yaygın orandır.


In [8]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # metin egitiyoruz
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 32,
    lora_alpha   = 64,
    lora_dropout = 0,
    bias         = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
    use_rslora   = False,
)
model.print_trainable_parameters()


trainable params: 73,400,320 || all params: 8,069,556,768 || trainable%: 0.9096


## 6 — Veri


In [9]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')

ds = load_dataset('fport/issue-writer-tr-en')

def to_text(r):
    t = tokenizer.apply_chat_template(r['messages'], tokenize=False,
                                      add_generation_prompt=False)
    # SFTTrainer tokenize ederken BOS'u kendisi ekler; sablonun bastaki <bos>'u
    # kalirsa dizide CIFT BOS olur ve egitim bozulur (Unsloth quickstart'ta da
    # ayni removeprefix var).
    return {'text': t.removeprefix('<bos>')}

ds = ds.map(to_text, remove_columns=['messages', 'meta'], num_proc=2)
print(ds)
print('\n--- bir ornegin ham hali (ilk 900 karakter) ---')
print(ds['train'][0]['text'][:900])


README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

data/train.jsonl: reconstructing file:   0%|          |  0.00B / 28.5MB            

data/train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/2.84M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.85M [00:00<?, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Map (num_proc=2):   0%|          | 0/10948 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1026 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1026 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10948
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1026
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1026
    })
})

--- bir ornegin ham hali (ilk 900 karakter) ---
<|turn>system
You are a Jira issue writer for a product engineering team. Convert the user's input into structured issue fields as JSON. Keep summaries under 80 characters and in the imperative mood. Do not add a type prefix to the summary. Put every unstated detail into `assumptions` or `clarifying_questions`.<turn|>
<|turn>user
Please write this up as a proper ticket.

---
Inside the story “Add recurring transfer to Transfers” we still need this one step done. Can you write the sub-task?
---<turn|>
<|turn>model
{
  "issue_type": "Sub-task",
  "summary": "Wire the feature flag and rollout configuration",
  "description": "h2. Objective\nWire the feature flag and rollout configuratio

### Uzunluk kontrolü

`MAXLEN`'i aşan örnek kırpılır ve kırpılan JSON kapanmaz — model bozuk çıktı üretmeyi
öğrenir. Oran %2'yi geçiyorsa `MAXLEN`'i büyüt.


In [13]:
import random

# Gemma 4'un "tokenizer"i aslinda multimodal bir Processor: ilk pozisyonel
# argumani `images`. tokenizer(metin) yazinca metin oraya gider, text=None kalir
# ve "'NoneType' object is not subscriptable" hatasi alirsin.
# Processor'in icindeki saf metin tokenizer'ini kullaniyoruz.
_tok = getattr(tokenizer, "tokenizer", tokenizer)

def n_tokens(text: str) -> int:
    ids = _tok(text)["input_ids"]
    if ids and isinstance(ids[0], list):      # bazi surumler batch dondurur
        ids = ids[0]
    return len(ids)

idx = random.Random(0).sample(range(len(ds["train"])), 500)
texts = ds["train"].select(idx)["text"]        # tek seferde okumak cok daha hizli
lens = sorted(n_tokens(t) for t in texts)

over = sum(x > MAXLEN for x in lens) / len(lens)
print(f"medyan {lens[len(lens)//2]} · p95 {lens[int(len(lens)*.95)]} · max {lens[-1]}")
print(f"MAXLEN={MAXLEN} asan oran: %{over*100:.1f}")
assert over < 0.05, "cok fazla kirpma var, MAXLEN artir"

# BOS kontrolu: dizinin basinda tam olarak BIR tane BOS olmali
ids = _tok(ds["train"][0]["text"])["input_ids"]
if ids and isinstance(ids[0], list):
    ids = ids[0]
bos = _tok.bos_token_id
n_bos = 0
for t in ids:
    if t == bos:
        n_bos += 1
    else:
        break
print(f"basta {n_bos} adet BOS ({_tok.bos_token})")
assert n_bos <= 1, "CIFT BOS: to_text icindeki removeprefix calismamis"
print("BOS tamam ✓")

medyan 582 · p95 1268 · max 1612
MAXLEN=2048 asan oran: %0.0
basta 0 adet BOS (<bos>)
BOS tamam ✓


## 7 — Eğitim öncesi çıktı (karşılaştırma için)

Aynı promptu eğitimden sonra tekrar soracağız.


In [17]:
from transformers import TextStreamer

# Processor'in icindeki saf metin tokenizer'i
_tok = getattr(tokenizer, "tokenizer", tokenizer)

# DIKKAT: bu metin egitim verisindeki sistem promptlarindan biriyle BIREBIR ayni.
# Cikarimda farkli bir prompt kullanirsan modeli dagilim disina cikarirsin;
# marka adi dahil hicbir kelimesini degistirme.
SYSTEM = ('Kıdemli bir çevik teslimat asistanısın. Ham ürün girdisini düzgün yazılmış '
          'Jira kayıtlarına çevirirsin. Yalnızca tek bir geçerli JSON nesnesi döndür, '
          'başka hiçbir şey yazma. INVEST ilkelerine uy, test edilebilir Given/When/Then '
          'kabul kriterleri yaz ve asla bilgi uydurma: girdide olmayan her şey '
          '`assumptions` ya da `clarifying_questions` alanına gider.')

USER = '''Bunu bir Jira kaydına çevir.

---
selam ekip, müşteriler fatura geçmişini tek tek açmak yerine toplu PDF olarak
indirmek istiyor. muhasebeciler ayda 30-40 fatura indiriyor, çok vakit alıyor.
bu sprint yetişir mi?
---'''


def encode(msgs):
    """Gemma 4'un Processor'i uc ayri yerde duz tokenizer gibi davranmiyor:

    - tokenizer(metin)         -> metin `images` argumanina gider
    - apply_chat_template(...) -> tokenize etmeden STRING doner
    - tokenize=True yolu       -> icerigin multimodal liste olmasini bekler
                                  ({"type": "text", ...}), duz string kabul etmez

    Bu yuzden egitimdeki yolu birebir tekrarliyoruz: once sablon metnini al,
    sonra metin tokenizer'i ile tokenize et. Sablon zaten <bos> ile basladigi
    icin add_special_tokens=False; aksi halde dizide cift BOS olur.
    """
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                         tokenize=False)
    return _tok(text, return_tensors="pt", add_special_tokens=False)


def ask(msgs, max_new=1200, stream=True):
    enc = encode(msgs).to("cuda")
    kw = dict(max_new_tokens=max_new, do_sample=False,
              pad_token_id=_tok.pad_token_id or _tok.eos_token_id)
    if stream:
        kw["streamer"] = TextStreamer(_tok, skip_prompt=True)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    n_in = enc["input_ids"].shape[1]
    return _tok.decode(out[0][n_in:], skip_special_tokens=True).strip()


# BOS kontrolu: cikarim dizisi de egitimdeki gibi tek BOS ile baslamali
_ids = encode([{"role": "user", "content": "test"}])["input_ids"][0].tolist()
_n_bos = 0
for t in _ids:
    if t == _tok.bos_token_id:
        _n_bos += 1
    else:
        break
print(f"cikarim dizisi basinda {_n_bos} BOS (1 olmali)")
assert _n_bos == 1, "BOS sayisi beklenenden farkli; add_special_tokens ayarini kontrol et"

MSGS = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": USER}]
before = ask(MSGS)


cikarim dizisi basinda 1 BOS (1 olmali)
```json
{
  "issue_type": "Story",
  "summary": "Müşterilerin fatura geçmişini toplu PDF olarak indirmesine olanak tanımak",
  "description": "Müşteriler, tek tek fatura geçmişlerini açmak yerine, belirli bir dönemdeki tüm faturaları toplu bir PDF dosyası olarak indirmek istiyorlar. Muhasebe departmanının aylık 30-40 fatura indirme süreci zaman alıcıdır.",
  "acceptance_criteria": [
    {
      "scenario": "Başarılı Toplu İndirme",
      "given": "Kullanıcı fatura geçmişi indirme ekranındadır.",
      "when": "Kullanıcı bir başlangıç ve bitiş tarihi seçer ve 'Toplu PDF İndir' butonuna tıklar.",
      "then": "Sistem, seçilen tarih aralığındaki tüm faturaları içeren tek bir PDF dosyası oluşturur ve kullanıcıya indirme bağlantısı sunar."
    },
    {
      "scenario": "Boş Dönem İndirme",
      "given": "Kullanıcı fatura geçmişi indirme ekranındadır.",
      "when": "Kullanıcı, hiç faturanın bulunmadığı bir tarih aralığı seçer ve 'Toplu PDF İndir' 

### 7b — Eğitim öncesi ölçüm (baseline)

Aynı fonksiyonu eğitimden sonra tekrar çağıracağız. **"İşe yaradı mı" sorusunun
tek dürüst cevabı bu iki tablonun yan yana konmasıdır.**

Baz model gözle bakınca makul bir issue üretiyor — ama şemaya uymuyor: `h2.`
bölümleri yok, kabul kriterlerinde `id` yerine uydurduğu alan var, `components`
ve `dor_check` hiç yok. İnsan için okunur, boru hattı için kullanılamaz.

25 örnek yaklaşık 8–12 dakika sürer. Eğitimden önce harcanan bu süre, sonunda
tahmin yerine sayı konuşmasını sağlar.


In [18]:
import json, re, collections
from datasets import load_dataset

_test = load_dataset('fport/issue-writer-tr-en', split='test')
_rows = [r for r in _test if r['meta']['task'] in ('draft_issue', 'bug_from_log')]

VER    = re.compile(r'\b\d+\.\d+(?:\.\d+)?\b')
PREFIX = re.compile(r'^\s*(\[(bug|story|task|epic)\]|(bug|story|task|epic)\s*[:\-])', re.I)
H2     = re.compile(r'^h2\. ', re.M)
REQ    = ('issue_type', 'summary', 'description', 'priority', 'labels',
          'components', 'dor_check')


def evaluate(n=25, label='model'):
    """Ayni metrikleri egitim oncesi ve sonrasi icin uretir.

    ONEMLI: Unsloth modeli EGITIM icin hazirlar — gradient checkpointing acik,
    KV cache kapali. Bu ayarlarla generate() her token icin butun diziyi yeniden
    hesaplar ve olcum saatlerce surer. for_inference() bunu duzeltir; olcum
    bitince modeli egitim moduna geri almak gerekir.

    json_strict  : cikti DOGRUDAN JSON mu (istedigimiz davranis)
    json_lenient : markdown fence/aciklama temizlenince JSON mu
    Ikisi arasindaki fark modelin ciktisini sarmalayip sarmalamadigini gosterir.
    """
    from unsloth import FastModel
    FastModel.for_inference(model)          # KV cache acilir, checkpointing kapanir

    agg = collections.defaultdict(list)
    for i, r in enumerate(_rows[:n], 1):
        raw  = ask(r['messages'][:2], max_new=1400, stream=False)
        gold = json.loads(r['messages'][2]['content'])

        try:
            p = json.loads(raw)
            agg['json_strict'].append(1)
        except json.JSONDecodeError:
            agg['json_strict'].append(0)
            cut = raw[raw.find('{'):raw.rfind('}') + 1]
            try:
                p = json.loads(cut)
            except Exception:
                agg['json_lenient'].append(0)
                continue
        agg['json_lenient'].append(1)

        agg['has_all_fields'].append(int(all(k in p for k in REQ)))
        agg['type_acc'].append(int(p.get('issue_type') == gold.get('issue_type')))

        s = p.get('summary', '')
        agg['summary_ok'].append(int(0 < len(s) <= 120 and not PREFIX.match(s)))
        agg['sections_ok'].append(int(len(H2.findall(p.get('description', ''))) >= 3))

        acs = p.get('acceptance_criteria') or []
        if p.get('issue_type') == 'Story':
            agg['ac_count_ok'].append(int(3 <= len(acs) <= 7))
        # Yalnizca kabul kriteri TASIYAN kayitlar uzerinden olc: Bug/Task/Epic
        # kayitlarini 0 saymak metrigi orneklem bilesimine bagimli yapar.
        if acs:
            agg['ac_shape_ok'].append(int(all(
                all(k in a for k in ('id', 'given', 'when', 'then')) for a in acs)))

        # uydurma: ciktidaki surum numaralari girdide de gecmeli
        agg['no_hallucination'].append(int(
            set(VER.findall(json.dumps(p, ensure_ascii=False)))
            <= set(VER.findall(r['messages'][1]['content']))))

        if i % 10 == 0:
            print(f'  {i}/{n}')

    # egitim moduna geri don, yoksa trainer.train() cok yavas calisir
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.train()

    print(f'\n=== {label} · {n} ornek ===')
    for k in sorted(agg):
        v = agg[k]
        print(f'  {k:18} %{sum(v) / len(v) * 100:5.1f}   ({sum(v)}/{len(v)})')
    return {k: sum(v) / len(v) for k, v in agg.items()}


BEFORE = evaluate(n=25, label='EGITIM ONCESI')


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

  10/25
  20/25

=== EGITIM ONCESI · 25 ornek ===
  ac_count_ok        % 33.3   (4/12)
  ac_shape_ok        %  0.0   (0/25)
  has_all_fields     %  0.0   (0/25)
  json_lenient       %100.0   (25/25)
  json_strict        %  0.0   (0/25)
  no_hallucination   %100.0   (25/25)
  sections_ok        %  0.0   (0/25)
  summary_ok         % 68.0   (17/25)
  type_acc           % 56.0   (14/25)


## 8 — Trainer

`max_steps` **yok** — tam epoch üzerinden gidiyoruz. `eval_steps` ile aşırı öğrenmeyi
izleyeceğiz: eval loss düşmeyi bırakıp yükselmeye başlarsa o noktadan sonrası ezber.


In [19]:
from trl import SFTTrainer, SFTConfig
import os

EPOCHS = 2.0

# Checkpoint dizini trainer olusturulmadan ONCE belli olmali: output_dir
# SFTConfig'e gecince sabitlenir, sonradan degistirmek ise yaramaz.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/issue-writer-gemma4-ckpt'
except Exception:
    CKPT = os.path.abspath('./checkpoints/gemma4')
os.makedirs(CKPT, exist_ok=True)
print('checkpoint dizini:', CKPT)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds['train'],
    eval_dataset  = ds['validation'].select(range(256)),
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAXLEN,
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = ACCUM,
        per_device_eval_batch_size  = BATCH,
        num_train_epochs = EPOCHS,
        learning_rate    = 1e-4,          # uzun egitimde 2e-4 fazla agresif
        lr_scheduler_type= 'cosine',
        warmup_ratio     = 0.03,
        optim            = 'adamw_8bit',
        weight_decay     = 0.01,
        logging_steps    = 20,
        eval_strategy    = 'steps', eval_steps = 200,
        save_strategy    = 'steps', save_steps = 200, save_total_limit = 2,
        output_dir       = CKPT,
        seed             = 3407,
        report_to        = 'none',
    ),
)
steps = int(len(ds['train']) * EPOCHS / (BATCH * ACCUM))
print(f'{len(ds["train"])} ornek · {EPOCHS} epoch · yaklasik {steps} adim')


Mounted at /content/drive


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


checkpoint dizini: /content/drive/MyDrive/issue-writer-gemma4-ckpt


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10948 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/256 [00:00<?, ? examples/s]

10948 ornek · 2.0 epoch · yaklasik 1368 adim


### Yalnızca cevaba loss uygula

Bu olmadan model senin **girdilerini de ezberler**. Gemma 4'ün tur işaretleri kullanılıyor.


In [20]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|turn>user\n',
    response_part    = '<|turn>model\n',
)
print('maskeleme uygulandi')


Map (num_proc=6):   0%|          | 0/10948 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/256 [00:00<?, ? examples/s]

maskeleme uygulandi


### Maske doğrulaması — bu hücreyi atlama

Eski notebook'ta yoktu. `instruction_part` / `response_part` metinleri modelin şablonuyla
birebir eşleşmezse maske sessizce boşa düşer ve saatlerce yanlış hedefe eğitim yaparsın.

Aşağıda **loss'a giren tokenlar** yazdırılıyor. Yalnızca JSON cevabını görmelisin;
sistem promptu veya kullanıcı mesajı görünüyorsa dur ve işaretleri düzelt.


In [21]:
ex = trainer.train_dataset[0]
ids    = ex['input_ids']
labels = ex['labels']

kept    = [i for i, l in zip(ids, labels) if l != -100]
ignored = [i for i, l in zip(ids, labels) if l == -100]

print(f'toplam {len(ids)} token · loss\'a giren {len(kept)} (%{len(kept)/len(ids)*100:.0f})')
print('\n=== LOSS HESAPLANAN KISIM (ilk 600 karakter) ===')
print(_tok.decode(kept)[:600])
print('\n=== MASKELENEN KISIM (ilk 400 karakter) ===')
print(_tok.decode(ignored)[:400])

assert 0.15 < len(kept) / len(ids) < 0.95, (
    'Maske oraninda anormallik var. Cok dusukse isaretler eslesmiyor, '
    'cok yuksekse maskeleme hic uygulanmamis demektir.')
print('\nmaske makul görünüyor ✓')


toplam 391 token · loss'a giren 273 (%70)

=== LOSS HESAPLANAN KISIM (ilk 600 karakter) ===
{
  "issue_type": "Sub-task",
  "summary": "Wire the feature flag and rollout configuration",
  "description": "h2. Objective\nWire the feature flag and rollout configuration\n\nh2. Context\nPart of the story “Add recurring transfer to Transfers”. Single-person step, roughly 2 hours.\n\nh2. Steps to Reproduce\n# Implement the change in the module noted above\n# Cover it with a test at the level it belongs to\n# Open the pull request linked to the parent story\n\nh2. Done When\n* flag toggles the behaviour without a deploy",
  "priority": "Medium",
  "severity": null,
  "labels": [
    "payment

=== MASKELENEN KISIM (ilk 400 karakter) ===
<|turn>system
You are a Jira issue writer for a product engineering team. Convert the user's input into structured issue fields as JSON. Keep summaries under 80 characters and in the imperative mood. Do not add a type prefix to the summary. Put every unstated de

## 9 — Eğitim

Checkpoint'ler Drive'a yazılır; oturum koparsa bu iki hücreyi tekrar çalıştır.

> **Loss hakkında:** Unsloth dokümanı E2B/E4B varyantlarında 13–15 bandındaki loss'un
> beklenen olduğunu belirtiyor (26B/31B'de 1–3). Yani **mutlak değere değil, eğilime**
> bak: `eval_loss` düşüyor mu? Asıl karar Adım 11'deki test metrikleriyle verilir.


In [22]:
import os

resume = os.path.isdir(CKPT) and any(
    d.startswith('checkpoint-') for d in os.listdir(CKPT))
print('kaldigi yerden devam' if resume else 'sifirdan basliyor')


sifirdan basliyor


In [23]:
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,948 | Num Epochs = 2 | Total steps = 1,370
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 73,400,320 of 8,069,556,768 (0.91% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
200,0.094286,0.200252
400,0.056220,0.229306
600,0.046316,0.270181
800,0.037083,0.280278
1000,0.037417,0.296442
1200,0.035566,0.301479
1370,0.038414,0.301429


Filter:   0%|          | 0/256 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-1200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-gemma4-ckpt/checkpoint-1370/tokenizer_config.json.


{'train_runtime': 6623.0646, 'train_samples_per_second': 3.306, 'train_steps_per_second': 0.207, 'total_flos': 5.628048603592397e+17, 'train_loss': 0.10910376284244287, 'epoch': 2.0}


## 10 — Eğitim sonrası aynı soru


In [24]:
import json
after = ask(MSGS, stream=False)
print('=== EGITIM SONRASI ===')
try:
    o = json.loads(after)
    print('JSON gecerli ✓')
    print('type    :', o.get('issue_type'))
    print('summary :', o.get('summary'))
    print('priority:', o.get('priority'), '| points:', o.get('story_points'))
    print('AC      :', len(o.get('acceptance_criteria', [])))
    print('varsayim:', o.get('assumptions'))
    print('sorular :', o.get('clarifying_questions'))
    print('\n' + o.get('description','')[:900])
except json.JSONDecodeError as e:
    print('JSON PARSE HATASI:', e); print(after[:1200])


=== EGITIM SONRASI ===
JSON gecerli ✓
type    : Story
summary : Fatura indirme ekle ve hata durumlarını kapsa
priority: Medium | points: 5
AC      : 5
varsayim: []
sorular : []

h2. Kullanıcı Hikâyesi
Bir muhasebesini yapan satıcı olarak fatura geçmişini tek tek açmak yerine toplu PDF olarak indirmek istiyorum; böylece muhasebeciler ayda 30-40 fatura indiriyor, çok vakit alıyor.

h2. Bağlam
Bu talep çeyreklik yol haritası incelemesinden çıktı. Bugün muhasebesini yapan satıcı bunu elle çözüyor ve her seferinde yaklaşık 12 dakika harcıyor.

h2. Kabul Kriterleri
*AC1 —* *Given* muhasebesini yapan satıcı bir tarih aralığı için fatura indirme talep ettiğinde — *When* dışa aktarma tamamlandığında — *Then* dosya adında dönem bilgisiyle iner ve ekrandakı toplamlarla birebir uyuşur
*AC2 —* *Given* dışa aktarım kişisel veri içerdiğinde — *When* dosya oluşturulduğunda — *Then* indirme linği 48 saat içinde geçersiz olur ve erişim denetim kaydına yazılır
*AC3 —* *Given* dışa aktarım 10.000 satırı a

## 11 — Eğitim sonrası ölçüm ve karşılaştırma

Aynı 25 örnek, aynı metrikler. Test seti eğitimde **hiç görülmemiş** içerik
çekirdeklerinden gelir; ezberi değil genellemeyi ölçer.

En büyük sıçramayı şema alanlarında beklemelisin — `sections_ok` ve
`ac_shape_ok` neredeyse sıfırdan doksanlara. `type_acc` daha az artar, çünkü baz
model Story/Bug ayrımını zaten kabaca yapabiliyor.

Hedef bantlar: `json_strict` > %95, `has_all_fields` > %95, `sections_ok` > %90,
`type_acc` > %85, `no_hallucination` > %90.

**`sections_ok` ve `ac_shape_ok` yükselmediyse** sorunu eğitimde değil maskede
ara: Adım 8'deki maske doğrulaması çıktısına dön.


In [25]:
AFTER = evaluate(n=25, label='EGITIM SONRASI')

print(f"\n{'metrik':18}{'once':>9}{'sonra':>9}{'fark':>9}")
for k in sorted(set(BEFORE) | set(AFTER)):
    b, a = BEFORE.get(k, 0) * 100, AFTER.get(k, 0) * 100
    arrow = '↑' if a - b > 1 else ('↓' if a - b < -1 else ' ')
    print(f'{k:18}{b:8.1f}%{a:8.1f}%{a - b:+8.1f} {arrow}')

# Daha genis olcum icin: evaluate(n=120, label='genis') — ~40 dakika.
# n=25'te bir orandaki guven araligi kabaca +-%12'dir, yani kucuk farklar
# gurultu olabilir; karar verecekseniz n'i 100'un uzerine cikarin.


  10/25
  20/25

=== EGITIM SONRASI · 25 ornek ===
  ac_count_ok        %100.0   (16/16)
  ac_shape_ok        % 64.0   (16/25)
  has_all_fields     %100.0   (25/25)
  json_lenient       %100.0   (25/25)
  json_strict        %100.0   (25/25)
  no_hallucination   % 76.0   (19/25)
  sections_ok        %100.0   (25/25)
  summary_ok         %100.0   (25/25)
  type_acc           %100.0   (25/25)

metrik                 once    sonra     fark
ac_count_ok           33.3%   100.0%   +66.7 ↑
ac_shape_ok            0.0%    64.0%   +64.0 ↑
has_all_fields         0.0%   100.0%  +100.0 ↑
json_lenient         100.0%   100.0%    +0.0  
json_strict            0.0%   100.0%  +100.0 ↑
no_hallucination     100.0%    76.0%   -24.0 ↓
sections_ok            0.0%   100.0%  +100.0 ↑
summary_ok            68.0%   100.0%   +32.0 ↑
type_acc              56.0%   100.0%   +44.0 ↑


## 12 — Kaydet


In [26]:
import os, shutil

OUT = os.path.abspath('./issue-writer-gemma4')
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print('adapter:', OUT)

# Colab'da Drive'a da kopyala; baska ortamda zaten kalici diskte
drive_dir = '/content/drive/MyDrive'
if os.path.isdir(drive_dir):
    dest = f'{drive_dir}/issue-writer-gemma4'
    shutil.copytree(OUT, dest, dirs_exist_ok=True)
    print('Drive:', dest)


Unsloth: Restored added_tokens_decoder metadata in /content/issue-writer-gemma4/tokenizer_config.json.


adapter: /content/issue-writer-gemma4
Drive: /content/drive/MyDrive/issue-writer-gemma4


In [27]:
# LoRA adapter'ini Hub'a yukle
model.push_to_hub('fport/issue-writer-gemma4-lora', token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub('fport/issue-writer-gemma4-lora', token=os.environ['HF_TOKEN'])


README.md:   0%|          | 0.00/567 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  294MB            

Saved model to https://huggingface.co/fport/issue-writer-gemma4-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp6322rxjn/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6322rxjn/tokenizer.json:  99%|#########9| 31.9MB / 32.2MB            

In [ ]:
# 16-bit birlestirilmis model (vLLM / dogrudan kullanim icin)
# model.push_to_hub_merged('fport/issue-writer-gemma4', tokenizer,
#                          save_method='merged_16bit', token=os.environ['HF_TOKEN'])

# GGUF (Ollama / llama.cpp icin) — donusum ~10-15 dk
# model.push_to_hub_gguf('fport/issue-writer-gemma4-gguf', tokenizer,
#                        quantization_method=['q4_k_m'], token=os.environ['HF_TOKEN'])


---

## Sorun giderme

**Maske doğrulaması patlıyor.** Gemma 4'ün tur işaretleri sürümle değişmiş olabilir.
Şunu çalıştırıp gerçek işaretleri gör, `train_on_responses_only` çağrısını ona göre düzelt:

```python
print(repr(tokenizer.apply_chat_template(
    [{'role':'user','content':'X'},{'role':'assistant','content':'Y'}],
    tokenize=False)))
```

**Kurulumda sürüm çakışması.** Ağustos 2026'da çalışan pinli set:

```python
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate xformers==0.0.34 peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
```

**OOM.** `BATCH=1`, `ACCUM=32`, `MAXLEN=1536`; hâlâ olmuyorsa `unsloth/gemma-4-E2B-it`.

**JSON bozuk çıkıyor.** Adım 6'daki kırpma oranına bak. `max_new_tokens` de yetersiz olabilir
— bizim çıktıların p95'i ~1.400 token.

**eval_loss yükseliyor.** Aşırı öğrenme. `EPOCHS=1` yap ya da en iyi checkpoint'e dön.

**Metrikler zayıf ama loss iyi.** Bu veri setinde tipik sebep: model şablonu ezberledi ama
alan seçimini öğrenmedi. `r`'yi 64'e çıkar veya 31B'ye geç.

## Sonuçları paylaş

Adım 11 tablosunu kaydet. Zayıf çıkan metrik varsa veri setinde o görevi hedefleyen
örnek sayısı artırılabilir — üretici `generator/` altında ve tohumlu.
